In [1]:
import IPython.core.pylabtools
import IPython.core.pylabtools
import os
import sys
import pandas as pd
import numpy as np
import datetime
import matplotlib.pyplot as plt
import mlflow
import keras_tuner as kt
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import GridSearchCV
from sklearn.ensemble import RandomForestRegressor
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, regularizers
import time
import itertools
from joblib import Parallel, delayed

# Ask TensorFlow to list all available physical GPUs
gpu_devices = tf.config.list_physical_devices('GPU')

if gpu_devices:
    print(f"✅ M3 Pro GPU ACTIVATED! Found: {gpu_devices}")
    # Optional: Set memory growth to prevent TF from hoarding all unified memory
    tf.config.experimental.set_memory_growth(gpu_devices[0], True)
else:
    print("❌ GPU not found. TensorFlow is falling back to the CPU.")

/opt/homebrew/Caskroom/miniforge/base/envs/ml-prod-v2/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


✅ M3 Pro GPU ACTIVATED! Found: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


# Data

In [ ]:
# Fix random seeds for reproducibility
np.random.seed(42)
tf.random.set_seed(42)

# Spark setup
from dotenv import load_dotenv
os.chdir(os.path.abspath(os.path.join(os.getcwd(), '../../')))
sys.path.append(os.getcwd())

from src.common.setup_spark import create_spark_session
from config.config_spark import Paths

# MLflow Setup
mlflow.set_tracking_uri("sqlite:///mlflow.db") # Local SQLite database for tracking
experiment_name = "SP500_Momentum_Backtest"
mlflow.set_experiment(experiment_name)
print(f"MLflow Experiment set to: {experiment_name}")

spark = create_spark_session()
print("Spark Session created.")

# Load Data
df_gold = spark.read.format("delta").load(Paths.SP500_MOMENTUM_VALUE_PROFITABLE_GROWTH_SURPRISE_CRASH_WEEKLY_GOLD)
df_gold.createOrReplaceTempView("gold_prices")

df = df_gold.toPandas()

df['date'] = pd.to_datetime(df['date'])
df['year'] = df['date'].dt.year
df['month'] = df['date'].dt.month
df['week'] = df['date'].dt.weekday

print(f"Data loaded: {df.shape}")
print(f"Years: {df['year'].unique().min()}")

2026-03-24 08:39:04.848 | INFO     | src.common.setup_spark:create_spark_session:19 - 🛠️ Configurant Spark avec le connecteur GCS : https://repo1.maven.org/maven2/com/google/cloud/bigdataoss/gcs-connector/hadoop3-2.2.6/gcs-connector-hadoop3-2.2.6-shaded.jar


MLflow Experiment set to: SP500_Momentum_Backtest


26/03/24 08:39:05 WARN Utils: Your hostname, MacBook-Pro-5.local resolves to a loopback address: 127.0.0.1; using 192.168.1.1 instead (on interface en0)
26/03/24 08:39:05 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Ivy Default Cache set to: /Users/forget/.ivy2/cache
The jars for the packages stored in: /Users/forget/.ivy2/jars
io.delta#delta-spark_2.12 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-25f03a6a-6383-46e7-9376-1042d714f4f3;1.0
	confs: [default]


:: loading settings :: url = jar:file:/opt/homebrew/Caskroom/miniforge/base/envs/ml-prod-v2/lib/python3.10/site-packages/pyspark/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


	found io.delta#delta-spark_2.12;3.2.1 in central
	found io.delta#delta-storage;3.2.1 in central
	found org.antlr#antlr4-runtime;4.9.3 in central
:: resolution report :: resolve 66ms :: artifacts dl 3ms
	:: modules in use:
	io.delta#delta-spark_2.12;3.2.1 from central in [default]
	io.delta#delta-storage;3.2.1 from central in [default]
	org.antlr#antlr4-runtime;4.9.3 from central in [default]
	---------------------------------------------------------------------
	|                  |            modules            ||   artifacts   |
	|       conf       | number| search|dwnlded|evicted|| number|dwnlded|
	---------------------------------------------------------------------
	|      default     |   3   |   0   |   0   |   0   ||   3   |   0   |
	---------------------------------------------------------------------
:: retrieving :: org.apache.spark#spark-submit-parent-25f03a6a-6383-46e7-9376-1042d714f4f3
	confs: [default]
	0 artifacts copied, 3 already retrieved (0kB/2ms)
26/03/24 08:39:05 

Spark Session created.


Data loaded: (547524, 166)
Years: 1990


In [12]:
df = df[df['bull_market_ma']==1]

In [13]:
class OHLCResampler:
    def __init__(self, data, date_col="Date", ticker_col="Ticker"):
        self.data = data.copy()
        self.date_col = date_col
        self.ticker_col = ticker_col

        # S'assurer que la colonne de date est bien en datetime
        self.data[date_col] = pd.to_datetime(self.data[date_col])
        self.data.set_index(date_col, inplace=True)

    def resample(self, period='W'):
        grouped = self.data.groupby(self.ticker_col)

        resampled = grouped.resample(period).agg({
            'Open': 'first',
            'High': 'max',
            'Low': 'min',
            'Close': 'last',
            'Volume': 'sum'
        }).reset_index()

        return resampled

# Backtest

## Generate Signals

In [14]:
def generate_signals(df, momentum_window, min_momentum, top_n, ma_fast_window, ma_slow_window, rebalance_freq='W'):
    """
    rebalance_freq : 'W' (Weekly), 'M' (Monthly), 'Q' (Quarterly), '6M' (Semi-Annually), 'Y' (Yearly)
    """
    data = df[['symbol', 'date', 'adjClose']].copy()
    data = data.sort_values(['symbol', 'date'])

    # --- 1. Indicateurs Techniques ---
    # Momentum (Rendement sur la fenêtre définie)
    data['Momentum'] = data.groupby('symbol')['adjClose'].pct_change(periods=momentum_window)

    # Moyennes Mobiles
    data['ma_fast'] = data.groupby('symbol')['adjClose'].transform(lambda x: x.rolling(ma_fast_window).mean())
    data['ma_slow'] = data.groupby('symbol')['adjClose'].transform(lambda x: x.rolling(ma_slow_window).mean())

    # Filtre de tendance (La MA rapide doit être au-dessus de la lente pour autoriser l'achat)
    data['Trend_Filter'] = data['ma_fast'] > data['ma_slow']

    # Rendement futur de la semaine à venir
    data['NextReturn'] = data.groupby('symbol')['adjClose'].shift(-1) / data['adjClose'] - 1

    # --- 2. GESTION DU REBALANCEMENT ---
    if rebalance_freq == 'W':
        data['is_rebalance_date'] = True
    else:
        # Création des périodes dynamiques
        if rebalance_freq == 'M':
            data['Period'] = data['date'].dt.to_period('M')
        elif rebalance_freq == 'Q':
            data['Period'] = data['date'].dt.to_period('Q')
        elif rebalance_freq == '6M':
            # Astuce pour les semestres (H1 / H2)
            data['Period'] = data['date'].dt.year.astype(str) + "H" + np.where(data['date'].dt.month <= 6, '1', '2')
        elif rebalance_freq == 'Y':
            data['Period'] = data['date'].dt.to_period('Y')
            
        # Trouver la dernière date de cotation pour chaque période
        rebalance_dates = data.groupby('Period')['date'].transform('max')
        data['is_rebalance_date'] = (data['date'] == rebalance_dates)

    # --- 3. RANKING & SÉLECTION ---
    # On isole uniquement les dates de rebalancement pour prendre les décisions
    reb_data = data[data['is_rebalance_date']].copy()
    
    # On filtre les actions qui respectent les conditions (Tendance OK + Momentum minimum)
    candidates = reb_data[(reb_data['Trend_Filter'] == True) & (reb_data['Momentum'] > min_momentum)].copy()
    
    # Classement des meilleures actions selon le Momentum
    candidates['Rank'] = candidates.groupby('date')['Momentum'].rank(method='first', ascending=False)
    
    # Sélection du Top N
    buys = candidates[candidates['Rank'] <= top_n].copy()
    buys['Target_Position'] = 1

    # On réintègre les signaux d'achat dans le dataset principal
    data = data.merge(buys[['symbol', 'date', 'Target_Position']], on=['symbol', 'date'], how='left')

    # Les jours de rebalancement où l'action n'est pas sélectionnée, la position cible devient 0
    data.loc[data['is_rebalance_date'] & data['Target_Position'].isna(), 'Target_Position'] = 0

    # PROPAGATION DU SIGNAL (Forward Fill)
    # L'action reste à 1 ou 0 jusqu'à la prochaine date de rebalancement
    data['Target_Position'] = data.groupby('symbol')['Target_Position'].ffill().fillna(0)

    # Nettoyage des lignes sans rendement futur connu
    data = data.dropna(subset=['NextReturn'])

    return data

## Vectorized Backtester

In [15]:
def run_vectorized_backtest(data, top_n, transaction_cost=0.001):
    """
    Backtest sans boucle For. Calcule tout le portefeuille via des opérations matricielles Pandas.
    """
    # 1. Calcul du poids de chaque action (Equi-pondéré sur le Top N)
    data['Weight'] = data['Target_Position'] / top_n
    
    # 2. Rendement généré par chaque action
    data['Strat_Return'] = data['Weight'] * data['NextReturn']
    
    # 3. Calcul du Turnover (Changement de poids) pour appliquer les frais
    # Si le poids passe de 0 à 1/Top_N, on paie les frais d'achat. Idem pour la vente.
    data['Weight_Change'] = data.groupby('symbol')['Weight'].diff().fillna(data['Weight'])
    data['Cost'] = data['Weight_Change'].abs() * transaction_cost
    
    # 4. Agrégation au niveau du Portefeuille par date
    port_returns = data.groupby('date')[['Strat_Return', 'Cost']].sum()
    port_returns['Net_Return'] = port_returns['Strat_Return'] - port_returns['Cost']
    
    # 5. Calcul de la courbe de capital
    port_returns['Capital'] = (1 + port_returns['Net_Return']).cumprod()
    
    return port_returns

## Running only one combination

In [16]:
def run_single_backtest(params, df_source, start_date, end_date):
    mom_win, min_mom, top_n, ma_fast, ma_slow, reb_freq = params

    default_output = {
        "Momentum_Window": mom_win, "Min_Momentum": min_mom, "Top_N": top_n,
        "MA_Fast": ma_fast, "MA_Slow": ma_slow, "Rebalance": reb_freq,
        "Total Return": np.nan, "CAGR": np.nan, "Max Drawdown": np.nan,
        "Sharpe Ratio": np.nan, "Error": None
    }

    try:
        # Si MA Rapide >= MA Lente, la combinaison est absurde, on l'ignore pour gagner du temps
        if ma_fast >= ma_slow:
            default_output["Error"] = "Invalid MA Config"
            return default_output

        # 1. Génération des signaux
        full_signals = generate_signals(
            df_source, momentum_window=mom_win, min_momentum=min_mom,
            top_n=top_n, ma_fast_window=ma_fast, ma_slow_window=ma_slow, rebalance_freq=reb_freq
        )

        # 2. Filtre des dates de test
        mask = (full_signals['date'] >= pd.to_datetime(start_date)) & (full_signals['date'] <= pd.to_datetime(end_date))
        backtest_data = full_signals.loc[mask]

        if backtest_data.empty:
            default_output["Error"] = "No data in timeframe"
            return default_output

        # 3. Lancement du Backtest
        res_df = run_vectorized_backtest(backtest_data, top_n=top_n, transaction_cost=0.001) # Frais de 0.1% par trade

        # 4. Calcul des métriques
        total_return = res_df['Capital'].iloc[-1] - 1
        n_years = (res_df.index[-1] - res_df.index[0]).days / 365.25
        cagr = (res_df['Capital'].iloc[-1]) ** (1 / max(1, n_years)) - 1
        
        rolling_max = res_df['Capital'].cummax()
        max_dd = ((res_df['Capital'] - rolling_max) / rolling_max).min()
        
        mean_ret = res_df['Net_Return'].mean()
        std_ret = res_df['Net_Return'].std()
        sharpe = (mean_ret / std_ret) * np.sqrt(52) if std_ret > 0 else 0 # Sqrt(52) car données hebdo

        # Mise à jour de l'output
        output = default_output.copy()
        output.update({
            "Total Return": total_return, "CAGR": cagr, 
            "Max Drawdown": max_dd, "Sharpe Ratio": sharpe
        })
        return output

    except Exception as e:
        default_output["Error"] = str(e)
        return default_output

## Grid Search

In [17]:
def grid_search_execution(df, param_grid, start_date, end_date):
    keys, values = zip(*param_grid.items())
    combinations = [v for v in itertools.product(*values)]

    print(f"🚀 Lancement de la Grid Search sur {len(combinations)} combinaisons...")
    start_time = time.time()

    # Exécution en parallèle sur tous les coeurs du processeur (n_jobs=-1)
    results_list = Parallel(n_jobs=-1)(
        delayed(run_single_backtest)(params, df, start_date, end_date) for params in combinations
    )

    end_time = time.time()
    print(f"✅ Terminé en {end_time - start_time:.2f} secondes.")

    results_df = pd.DataFrame(results_list)
    
    # On affiche les 10 meilleures stratégies triées par Sharpe Ratio
    best_strats = results_df[results_df['Error'].isna()].sort_values(by='Sharpe Ratio', ascending=False)
    return best_strats

## Running Backtest

In [36]:
param_grid = {
    'momentum_window': [12, 26, 52],            # 3 mois, 6 mois, 1 an
    'min_momentum': [0, 0.05, 0.10, 0.20],         # Rendement min passé : 5%, 10%, 20%
    'top_n': [10],                       # Concentré vs Diversifié
    'ma_fast': [15],                        # Trend court terme
    'ma_slow': [26],                       # Trend long terme (1 an, 2 ans)
    'rebalance_freq': ['W', 'M', 'Q', '6M']     # Hebdo, Mensuel, Trimestriel, Semestriel
}

# Assure-toi que ton dataframe 'df' est bien chargé avant de lancer ça !
best_strategies_df = grid_search_execution(
    df=df,
    param_grid=param_grid,
    start_date='2013-01-01',
    end_date='2026-01-01'
)

print("\n🏆 Top 10 des meilleures configurations :")

🚀 Lancement de la Grid Search sur 48 combinaisons...


/var/folders/6k/82j2nnl13hj9ld7nmzpdt6fh0000gn/T/ipykernel_2786/3711529127.py:6: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
/var/folders/6k/82j2nnl13hj9ld7nmzpdt6fh0000gn/T/ipykernel_2786/3711529127.py:9: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
/var/folders/6k/82j2nnl13hj9ld7nmzpdt6fh0000gn/T/ipykernel_2786/3711529127.py:13: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the document

✅ Terminé en 3.83 secondes.

🏆 Top 10 des meilleures configurations :


In [37]:
best_strategies_df.sort_values('CAGR', ascending=False).head(20)

,Momentum_Window,Min_Momentum,Top_N,MA_Fast,MA_Slow,Rebalance,Total Return,CAGR,Max Drawdown,Sharpe Ratio,Error
30,26,0.20,10,15,26,Q,9.861297,0.201818,-0.221125,1.054873,None
26,26,0.10,10,15,26,Q,9.861297,0.201818,-0.221125,1.054873,None
22,26,0.05,10,15,26,Q,9.861297,0.201818,-0.221125,1.054873,None
18,26,0.00,10,15,26,Q,9.861297,0.201818,-0.221125,1.054873,None
17,26,0.00,10,15,26,M,8.705466,0.191441,-0.256930,1.068437,None
25,26,0.10,10,15,26,M,8.705466,0.191441,-0.256930,1.068437,None
21,26,0.05,10,15,26,M,8.705466,0.191441,-0.256930,1.068437,None
29,26,0.20,10,15,26,M,8.705466,0.191441,-0.256930,1.068437,None
42,52,0.10,10,15,26,Q,7.138047,0.175376,-0.210657,0.996191,None
38,52,0.05,10,15,26,Q,7.138047,0.175376,-0.210657,0.996191,None


In [38]:
best_strategies_df.sort_values('Sharpe Ratio', ascending=False).head(20)

,Momentum_Window,Min_Momentum,Top_N,MA_Fast,MA_Slow,Rebalance,Total Return,CAGR,Max Drawdown,Sharpe Ratio,Error
17,26,0.00,10,15,26,M,8.705466,0.191441,-0.256930,1.068437,None
21,26,0.05,10,15,26,M,8.705466,0.191441,-0.256930,1.068437,None
29,26,0.20,10,15,26,M,8.705466,0.191441,-0.256930,1.068437,None
25,26,0.10,10,15,26,M,8.705466,0.191441,-0.256930,1.068437,None
26,26,0.10,10,15,26,Q,9.861297,0.201818,-0.221125,1.054873,None
22,26,0.05,10,15,26,Q,9.861297,0.201818,-0.221125,1.054873,None
18,26,0.00,10,15,26,Q,9.861297,0.201818,-0.221125,1.054873,None
30,26,0.20,10,15,26,Q,9.861297,0.201818,-0.221125,1.054873,None
42,52,0.10,10,15,26,Q,7.138047,0.175376,-0.210657,0.996191,None
38,52,0.05,10,15,26,Q,7.138047,0.175376,-0.210657,0.996191,None
